# Atelier guidé · De la vente à la synthèse

Suivez les six leçons. Les cellules de démonstration fonctionnent dès le départ ; les cellules « À vous » sont à compléter. « Tout exécuter » relance le carnet dans l’ordre. Cet atelier n’est pas noté et ne remplace pas le TP. Exportez votre carnet pour le conserver, puis réimportez-le pour reprendre.

Nous utilisons un petit fichier de fournitures de bureau, distinct des ventes de matériel informatique du TP.

## 1 · Lire
La chaîne ci-dessous contient le fichier complet. StringIO permet à read_csv de le lire comme un fichier. Observez les colonnes et la présence répétée de A01.

In [ ]:
from io import StringIO
import pandas as pd
import matplotlib.pyplot as plt
texte_csv = 'vente_id;region;produit;quantite;prix_unitaire\nA01;Nord;Carnet;2;5\nA02;Sud;Stylo;5;2\nA03; Nord ;Carnet;3;5\nA04;Est;Stylo;inconnu;2\nA05;;Carnet;1;5\nA06;Sud;Stylo;0;2\nA01;Nord;Carnet;2;5\n'
atelier = pd.read_csv(StringIO(texte_csv), sep=";")
print(atelier.head(3))

### À vous · Lire les dernières lignes
Remplacez None par une expression qui renvoie les deux dernières lignes. Les corrections expliquées se trouvent dans la leçon 1.

In [ ]:
dernieres_lignes = None  # À compléter
print("Exercice 1 à compléter" if dernieres_lignes is None else dernieres_lignes)

## 2 · Diagnostiquer
Les types, les absences et les doublons répondent à trois questions différentes. Regardez le type de quantite avant toute conversion.

In [ ]:
print("Dimensions :", atelier.shape)
print(atelier.dtypes)
print("Absences :", atelier.isna().sum().to_dict())
print("Identifiants répétés :", atelier.loc[atelier["vente_id"].duplicated(False), "vente_id"].tolist())

### À vous · Compter les ventes uniques
Remplacez None par le nombre d’identifiants distincts et comparez-le au nombre de lignes.

In [ ]:
identifiants_uniques = None  # À compléter
print("Exercice 2 à compléter" if identifiants_uniques is None else ("Correct : six ventes distinctes" if identifiants_uniques == 6 else "Revoir la différence entre lignes et identifiants distincts"))

## 3 · Filtrer
Un masque sélectionne des lignes. Ce premier filtre agit sur les données brutes : A01 apparaît encore deux fois.

In [ ]:
masque = atelier["produit"].eq("Carnet")
print(atelier.loc[masque, ["vente_id", "region", "produit"]])

exemple = pd.DataFrame({"region": ["Nord", "Sud", "Nord", "Est"], "quantite": [2, 5, 4, 1]})
masque_combine = (exemple["region"] == "Nord") & (exemple["quantite"] >= 3)
print("Exemple de deux conditions :", masque_combine.tolist())
print(exemple.loc[masque_combine])

### À vous · Combiner deux conditions
Dans exemple, sélectionnez les lignes du Sud OU de quantité inférieure à trois. Écrivez les positions attendues avant le calcul. Que se passe-t-il avec ET ? Consultez la correction de la leçon 3.

In [ ]:
selection = None  # À compléter avec exemple.loc[...]
print("Exercice 3 à compléter" if selection is None else selection)

## 4 · Nettoyer et tracer
On retire une copie exacte, on contrôle l’unicité, puis on convertit et on applique les règles de l’atelier. Les rejets restent consultables.

In [ ]:
doublons = atelier.loc[atelier.duplicated()].copy()
travail = atelier.drop_duplicates().copy()
assert travail["vente_id"].is_unique
for colonne in ["quantite", "prix_unitaire"]:
    travail[colonne] = pd.to_numeric(travail[colonne], errors="coerce")
valide = travail["quantite"].notna() & travail["quantite"].gt(0) & travail["quantite"].mod(1).eq(0) & travail["prix_unitaire"].notna() & travail["prix_unitaire"].gt(0)
rejets = travail.loc[~valide].copy()
propre = travail.loc[valide].copy()
propre["region"] = propre["region"].astype("string").str.strip().replace("", pd.NA).fillna("Non renseignée")
print("Doublons retirés :", len(doublons))
print("Rejets :", rejets["vente_id"].tolist())
print(propre)

### À vous · Réconcilier les comptes
Écrivez une égalité reliant les effectifs de atelier, doublons, rejets et propre. Vérifiez qu’elle vaut True.

In [ ]:
bilan_coherent = None  # À compléter
print("Exercice 4 à compléter" if bilan_coherent is None else bilan_coherent)

## 5 · Calculer et agréger
Les noms des colonnes décrivent les indicateurs et empêchent de confondre ventes, articles et euros.

In [ ]:
propre["montant"] = propre["quantite"] * propre["prix_unitaire"]
total = propre["montant"].sum()
par_produit = propre.groupby("produit").agg(montant=("montant", "sum"), ventes=("vente_id", "count"), articles=("quantite", "sum")).sort_values("montant", ascending=False)
par_region = propre.groupby("region")["montant"].sum()
assert par_produit["montant"].sum() == par_region.sum() == total
print(par_produit)
print(par_region)
print("Montant total :", total)

### À vous · Choisir le dénominateur
Calculez le pourcentage du montant représenté par les carnets. Dans une cellule texte, distinguez le montant moyen par vente du prix moyen par article.

In [ ]:
part_carnets = None  # À compléter, résultat en pourcentage
print("Exercice 5 à compléter" if part_carnets is None else ("Correct : 75 %" if abs(part_carnets - 75) < .001 else "Divisez le montant des carnets par le total, puis multipliez par 100"))

## 6 · Visualiser
Le titre précise le périmètre ; l’axe indique l’unité et commence à zéro. Le tableau accompagne toujours le graphique.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
par_produit["montant"].plot.bar(ax=ax, color="#1a2644")
ax.set(title="Montant des ventes valides par produit", xlabel="Produit", ylabel="Montant en euros")
ax.set_ylim(bottom=0)
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()
print(par_produit)

### À vous · Expliquer le résultat
Ajoutez ci-dessous un graphique par région et son tableau. Modifiez ensuite la conclusion pour donner un constat chiffré, une limite et une vérification à effectuer.

In [ ]:
# À compléter : graphique de par_region et tableau des valeurs
print("Exercice 6 : graphique régional à compléter")

## Ma conclusion
À compléter : observation chiffrée, limite des données et prochaine vérification.

Quand vous avez terminé, retournez à la leçon 6 puis passez au TP de synthèse. Exportez cet atelier si vous souhaitez garder vos essais.